# 03 Run Experiments

Objective: run both tasks for selected models, cache every raw response, and preserve invalid outputs for auditability.

The provider-aware CLI runner is the recommended path for GPU/server runs. The notebook cells below remain available for direct OpenAI-compatible local runs.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu
eu = importlib.reload(eu)

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)
DATASET_ID = eu.normalize_dataset_id(os.getenv("DATASET_ID", "nice"))
DATASET_SUFFIX = eu.dataset_suffix(DATASET_ID)
BENCHMARK_VARIANT = os.getenv("BENCHMARK_VARIANT", "must").strip().lower()
VARIANT_SUFFIX = eu.variant_suffix(BENCHMARK_VARIANT)
ARTIFACT_SUFFIX = eu.dataset_variant_suffix(DATASET_ID, BENCHMARK_VARIANT)

PROJECT_ROOT, CONFIG_PATH, DATASET_ID, BENCHMARK_VARIANT


## Provider-Aware CLI Runner


In [ ]:
import shlex

RUN_CONFIG = Path(os.getenv("RUN_CONFIG", PROJECT_ROOT / "run_configs/current_run.json"))
profile_hint = os.getenv("RUN_PROFILE", "local_llama_cpp")
model_hint = os.getenv("RUN_MODEL", "")
mode_hint = os.getenv("RUN_MODE", "smoke")

command = [
    ".venv/bin/python",
    "scripts/run_experiment_from_config.py",
    "--config",
    str(RUN_CONFIG),
    "--profile",
    profile_hint,
    "--dataset",
    DATASET_ID,
    "--variant",
    BENCHMARK_VARIANT,
    "--mode",
    mode_hint,
]
if model_hint:
    command.extend(["--model", model_hint])

print("Recommended server command:")
print(" ".join(shlex.quote(part) for part in command))
print()
print("Copy run_configs/full_matrix.example.json to run_configs/current_run.json and edit it for the provider/model matrix.")
print("For llama.cpp, start the server with one model, then pass --model for that loaded model.")
print("Set batch_size in the selected provider profile to reduce API calls while preserving one JSONL row per item/sample.")


## Configure Run


In [ ]:
HOST = os.getenv("HOST", CONFIG["llm"]["host"])
MODELS = [m.strip() for m in os.getenv("MODELS", ",".join(CONFIG["llm"]["models"])).split(",") if m.strip()]
RUN_FULL_EXPERIMENT = os.getenv("RUN_FULL_EXPERIMENT", "true").lower() in {"1", "true", "yes"}
deterministic = CONFIG["llm"]["deterministic"]
stochastic = CONFIG["llm"]["stochastic"]
REQUEST_CONCURRENCY = eu.resolve_llm_concurrency(CONFIG)
SAVE_PRELIMINARY_RESULTS = os.getenv("SAVE_PRELIMINARY_RESULTS", "true").lower() in {"1", "true", "yes"}
PRELIMINARY_EVERY_N_CALLS = max(1, int(os.getenv("PRELIMINARY_EVERY_N_CALLS", "50")))
output_path = eu.artifact_path(PROJECT_ROOT / "data/processed/model_outputs_raw.jsonl", DATASET_ID, BENCHMARK_VARIANT)
run_id = eu.new_run_id("full" if BENCHMARK_VARIANT == "must" else f"full-{BENCHMARK_VARIANT}")

benchmark_path = eu.artifact_path(PROJECT_ROOT / "data/processed/benchmark_items.csv", DATASET_ID, BENCHMARK_VARIANT)
benchmark = eu.read_csv_rows(benchmark_path)
planned_calls = len(benchmark) * len(MODELS) * 2 * (1 + int(stochastic["samples"]))
print({
    "HOST": HOST,
    "MODELS": MODELS,
    "RUN_FULL_EXPERIMENT": RUN_FULL_EXPERIMENT,
    "REQUEST_CONCURRENCY": REQUEST_CONCURRENCY,
    "SAVE_PRELIMINARY_RESULTS": SAVE_PRELIMINARY_RESULTS,
    "PRELIMINARY_EVERY_N_CALLS": PRELIMINARY_EVERY_N_CALLS,
    "run_id": run_id,
    "BENCHMARK_VARIANT": BENCHMARK_VARIANT,
})
print(f"Benchmark path: {benchmark_path}")
print(f"Benchmark items: {len(benchmark)}")
print(f"Planned calls: {planned_calls}")


## Run Full Experiment


In [ ]:
task1_template = eu.load_prompt(PROJECT_ROOT / "prompts/mandatory_entailment.txt")
task2_template = eu.load_prompt(PROJECT_ROOT / "prompts/modality_extraction.txt")

def prompt_for(task, item):
    if task == "task1":
        return eu.render_prompt(
            task1_template,
            source_statement=item["source_statement"],
            candidate_requirement=item["candidate_requirement"],
        )
    if task == "task2":
        return eu.render_prompt(task2_template, source_statement=item["source_statement"])
    raise ValueError(task)

def request_job(
    item,
    task,
    model,
    sample_kind,
    sample_index,
    temperature,
    top_p,
    run_id,
    request_index,
    prompt=None,
    prompt_version=None,
):
    prompt = prompt if prompt is not None else prompt_for(task, item)
    return {
        "request_index": request_index,
        "run_id": run_id,
        "model": model,
        "host": HOST,
        "task": task,
        "item": item,
        "sample_index": sample_index,
        "sample_kind": sample_kind,
        "temperature": temperature,
        "top_p": top_p,
        "prompt_version": prompt_version or CONFIG["project"]["prompt_version"],
        "prompt": prompt,
        "max_tokens": int(CONFIG["llm"]["max_tokens"]),
        "timeout_s": int(CONFIG["llm"]["timeout_s"]),
        "api_key_env": CONFIG["llm"]["api_key_env"],
    }


In [ ]:
if RUN_FULL_EXPERIMENT:
    total = 0
    last_snapshot_at = 0
    jobs = []

    def write_preliminary_snapshot(total_calls):
        run_rows = [row for row in eu.read_jsonl(output_path) if row.get("run_id") == run_id]
        snapshot = eu.write_preliminary_result_snapshot(
            benchmark,
            run_rows,
            PROJECT_ROOT,
            variant=BENCHMARK_VARIANT,
            dataset_id=DATASET_ID,
            expected_stochastic_samples=int(stochastic["samples"]),
        )
        print(
            f"Saved preliminary snapshot after {total_calls}/{planned_calls} calls: "
            f"{snapshot['summary_rows']} summary rows, {snapshot['progress_rows']} progress rows"
        )
        print(f"Preliminary table: {snapshot['paths']['table']}")

    for model in MODELS:
        for item in benchmark:
            for task in ["task1", "task2"]:
                jobs.append(request_job(
                    item=item,
                    task=task,
                    model=model,
                    sample_kind="deterministic",
                    sample_index=0,
                    temperature=float(deterministic["temperature"]),
                    top_p=float(deterministic["top_p"]),
                    run_id=run_id,
                    request_index=len(jobs),
                ))
                for sample_index in range(int(stochastic["samples"])):
                    jobs.append(request_job(
                        item=item,
                        task=task,
                        model=model,
                        sample_kind="stochastic",
                        sample_index=sample_index,
                        temperature=float(stochastic["temperature"]),
                        top_p=float(stochastic["top_p"]),
                        run_id=run_id,
                        request_index=len(jobs),
                    ))
    assert len(jobs) == planned_calls

    print(f"Dispatching {len(jobs)} full-experiment calls with concurrency={REQUEST_CONCURRENCY}")
    for record in eu.run_completion_jobs(jobs, max_workers=REQUEST_CONCURRENCY):
        eu.append_jsonl(output_path, record)
        total += 1
        if total % 100 == 0 or total == planned_calls:
            print(f"Completed {total}/{planned_calls} calls")
        if SAVE_PRELIMINARY_RESULTS and total - last_snapshot_at >= PRELIMINARY_EVERY_N_CALLS:
            write_preliminary_snapshot(total)
            last_snapshot_at = total
    if SAVE_PRELIMINARY_RESULTS:
        write_preliminary_snapshot(total)
    print(f"Done. Wrote records to {output_path}")
else:
    print("Full experiment not run. Set RUN_FULL_EXPERIMENT=true after the pilot gate passes.")


## Parse-Failure Audit


In [ ]:
all_rows = eu.read_jsonl(output_path)
run_rows = [row for row in all_rows if row.get("run_id") == run_id]
if run_rows:
    status_counts = {}
    for row in run_rows:
        status_counts[row["parse_status"]] = status_counts.get(row["parse_status"], 0) + 1
    print(status_counts)
    print(f"Parse success rate: {status_counts.get('ok', 0) / len(run_rows):.3f}")
else:
    print("No rows for this run_id yet.")


## Preliminary Snapshot Files


In [ ]:
paths = eu.preliminary_result_paths(PROJECT_ROOT, BENCHMARK_VARIANT, dataset_id=DATASET_ID)
for name, path in paths.items():
    print(f"{name}: {path} ({'exists' if path.exists() else 'missing'})")
